# Observation-masked NPE — Case study II: *Xylella fastidiosa* in Apulia

Reproduces the *Xylella fastidiosa* results of:

> Retkute R. & Gilligan C.A. *Observation-masked neural posterior estimation for
> heterogeneous epidemiological surveillance data.*

| Output | Paper item |
|---|---|
| `figure3a.pdf` | Figure 3A — posterior-predictive fits, 17 olive groves |
| `figure3b.pdf` | Figure 3B — predicted state proportions at year 16 |
| `figure3c.pdf` | Figure 3C — forecast variance vs extrapolation horizon |
| `figure4a.pdf` | Figure 4A — all 127 survey schedules ranked by forecast variance |
| `figure4b.pdf` | Figure 4B — best schedule at each observation count |
| `table4.csv` | Table 4 — parameters compared with White *et al.* (2020) |
| `figure_si3.pdf` | SI Figure 3 — posteriors vs grove size |
| `figure_si4a.pdf` | SI Figure 4A — simulation-based calibration rank CDFs |
| `figure_si4b.pdf` | SI Figure 4B — TARP coverage |

**Model.** Discrete-time annual compartmental model of White *et al.* (2020),
$S \to I_A \to I_S \to I_D$, simulated stochastically on $N$ trees (paper eqns
2.22–2.23). The desiccation delay is fixed at $\tau = 3$ years, giving
$\tau + 1$ annual sojourn stages in $I_S$. Six parameters are inferred:
$\theta = (\beta, b_A, b_D, T_A, T_D, I_{A,0})$.

**Observation masking.** Each grove's annual severity observations are placed on
a common 7-year grid (years 4–10 after epidemic establishment) with a binary
mask, giving the 29-dimensional network input of paper eqn 3.3:
`[y_1 m_1, ..., y_10 m_10, m_1, ..., m_10, N/N_max]`, where each $y$ holds the
three observed proportions $((S+I_A)/N,\ I_S/N,\ I_D/N)$.

**Data.** `data/Xf_epi.csv` (per-grove disease proportions by year and state) and
`data/Xf_number_trees.csv` (trees surveyed per grove), from White *et al.* (2020).

**Runtime.** Simulation and training take roughly 45 minutes on a single CPU.
Set `QUICK = True` for a fast smoke test (results will not match the paper).

## 1. Imports and configuration

In [ ]:
import itertools
import os
import pickle
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import Normalize
from scipy import stats as sstats
from sbi.utils import BoxUniform

try:
    from sbi.inference import NPE
except ImportError:                      # older sbi releases
    from sbi.inference import SNPE as NPE

QUICK = False   # True -> tiny training/diagnostic budgets for a smoke test

DATA_PATH = "data/Xf_epi.csv"
TREES_PATH = "data/Xf_number_trees.csv"

# --- model structure -----------------------------------------------------------
TAU = 3               # desiccation delay (years), fixed: tau + 1 annual I_S stages
N_YEARS = 16          # annual steps 0..16
CANON_STATES = ["S+IA", "IS", "ID"]

# --- grove size ----------------------------------------------------------------
N_MIN, N_MAX = 25, 600        # training range for trees per grove (data: 31-521)
N_SCALE = float(N_MAX)

# --- inference budgets ---------------------------------------------------------
N_TRAIN = 8_000 if QUICK else 1_000_000
N_POST = 100                              # posterior draws per observed record
N_SBC = 200 if QUICK else 10_000          # SBC / TARP trials
L_SBC = N_POST                            # posterior draws per SBC trial
CHUNK = 500                               # batch size for chunked TARP sampling
MASK_MIX_P = 0.5                          # P(real grove schedule) vs P(random subset)

SEED = 0

# --- prior ---------------------------------------------------------------------
# Ranges as in SI Table S2. beta and IA0 are bounded by per-grove least-squares
# fits to the data, which place beta in ~2-48 and IA0 below ~0.07 for every grove.
PARAMS = ["beta", "bA", "bD", "TA", "TD", "IA0"]
PRIORS = {"beta": (0.0, 50.0), "bA": (0.0, 1.0), "bD": (0.0, 1.0),
          "TA": (0.75, 1.5), "TD": (0.0, 12.0), "IA0": (0.0, 0.1)}
LOW = np.array([PRIORS[p][0] for p in PARAMS])
HIGH = np.array([PRIORS[p][1] for p in PARAMS])

PARAM_LABELS = {"beta": r"$\beta$", "bA": r"$b_A$", "bD": r"$b_D$",
                "TA": r"$T_A$", "TD": r"$T_D$", "IA0": r"$I_{A,0}$"}

STATE_COLORS = {"S+IA": "tab:blue", "IS": "tab:green", "ID": "gold"}
YEARS_GRID = np.arange(N_YEARS + 1)
YR16 = 16                      # forecast horizon used in Figures 3B, 3C and 4
ID_K = CANON_STATES.index("ID")

torch.manual_seed(SEED)
print(f"Estimated parameters: {PARAMS}  | tau fixed = {TAU}  | "
      f"N_TRAIN = {N_TRAIN:,}  | QUICK = {QUICK}")

## 2. Simulator: stochastic discrete-time annual model

Annual transition probabilities (paper eqn 2.23):

$$\alpha_t = 1 - \exp\!\left[-\beta\,\frac{I_S + b_A I_A + b_D I_D}{N}\right],
  \qquad \sigma = 1 - e^{-1/T_A}, \qquad \gamma = 1 - e^{-1/T_D}.$$

Transitions are Binomial draws on integer tree counts, so posterior uncertainty
reflects finite-population stochasticity at each grove's actual size.

In [ ]:
def simulate_curve(beta, bA, bD, TA, TD, IA0, rng, N, tau=TAU, n_years=N_YEARS):
    """Simulate one epidemic on N trees.

    Returns
    -------
    (n_years + 1, 3) array
        Proportions [(S + I_A)/N, I_S/N, I_D/N] at years 0..n_years; rows sum to 1.
    """
    L = tau + 1
    IA = min(N, max(1, int(round(N * IA0)))) if IA0 > 0 else 0
    S = N - IA
    Is = np.zeros(L, dtype=int)
    ID = 0

    out = np.empty((n_years + 1, 3))
    for t in range(n_years + 1):
        sympt = int(Is.sum())
        out[t] = [(S + IA) / N, sympt / N, ID / N]

        alpha = 1 - np.exp(-beta * (sympt + bA * IA + bD * ID) / N)
        sigma = 1 - np.exp(-1.0 / TA)
        gamma = 1 - np.exp(-1.0 / TD)

        new_inf = rng.binomial(S, alpha)                    # S     -> I_A
        new_sym = rng.binomial(IA, sigma)                   # I_A   -> I_S1
        new_des = rng.binomial(int(Is[L - 1]), gamma)       # I_Slast -> I_D

        old = Is.copy()
        newIs = np.zeros(L, dtype=int)
        newIs[0] = new_sym
        for j in range(1, L - 1):
            newIs[j] = old[j - 1]                           # annual pass-through
        if L >= 2:
            newIs[L - 1] = old[L - 1] - new_des + old[L - 2]
        else:
            newIs[0] = old[0] - new_des + new_sym

        S = S - new_inf
        IA = IA - new_sym + new_inf
        Is = newIs
        ID = ID + new_des
    return out

## 3. Survey data

Visual assessments of canopy desiccation for 2,959 olive trees in 17 groves,
scored 0–5 and grouped into three compartments: score 0 as susceptible or
symptomless ($S + I_A$), scores 1–3 as symptomatic ($I_S$), and scores 4–5 as
desiccated ($I_D$). Each grove was surveyed at only two or three time points
(paper Table 2).

In [ ]:
def load_plots(path=DATA_PATH):
    """Load per-grove observed proportions, ordered by survey year."""
    df = pd.read_csv(path)
    plots, positional = {}, []
    for plot, g in df.groupby("Plot"):
        years, obs = [], []
        for yr, gy in g.groupby("Year"):
            st = gy.groupby("State")["Proportion_of_trees"].sum()
            if all(s in st.index for s in CANON_STATES):
                v = np.array([st[s] for s in CANON_STATES], float)
            else:
                # A few grove-years carry a single repeated state label. The values
                # are still in CSV row order (S+IA, IS, ID); any absent trailing
                # state is taken as zero.
                vals = gy["Proportion_of_trees"].to_numpy(float)
                v = np.zeros(3)
                v[:len(vals)] = vals[:3]
                positional.append((plot, int(yr)))
            if v.sum() <= 0:
                continue
            years.append(int(yr))
            obs.append(v / v.sum())
        if years:
            order = np.argsort(years)
            plots[plot] = {"years": np.array(years)[order],
                           "obs": np.array(obs)[order]}
    print(f"Loaded {len(plots)} groves. Grove-years read positionally: {positional}")
    return plots


plots = load_plots()

trees = pd.read_csv(TREES_PATH)
tree_col = [c for c in trees.columns if c.lower() != "plot"][0]
N_by_plot = dict(zip(trees["plot"], trees[tree_col].astype(int)))
for p in plots:
    plots[p]["N"] = int(N_by_plot[p])

YEAR_GRID = np.array(sorted({int(y) for d in plots.values() for y in d["years"]}))
YEAR_IDX = {int(y): i for i, y in enumerate(YEAR_GRID)}
PATTERNS = [d["years"] for d in plots.values()]

print(f"Trees per grove: {min(N_by_plot.values())}-{max(N_by_plot.values())}  "
      f"(total {sum(N_by_plot[p] for p in plots):,})")
print(f"Survey-year grid: {list(YEAR_GRID)}")
print(pd.DataFrame([{"plot": p, "N": d["N"], "years": list(d["years"])}
                    for p, d in sorted(plots.items(), key=lambda kv: -kv[1]["N"])])
      .to_string(index=False))

## 4. Observation-masked input

In [ ]:
def build_input(prop_rows, years_obs, N):
    """Masked network input: [proportions * mask, mask, N / N_SCALE]."""
    x = np.zeros(len(YEAR_GRID) * 3)
    m = np.zeros(len(YEAR_GRID))
    for yr, p in zip(years_obs, prop_rows):
        i = YEAR_IDX[int(yr)]
        x[3 * i:3 * i + 3] = p
        m[i] = 1.0
    return np.concatenate([x, m, [N / N_SCALE]])


INPUT_DIM = len(YEAR_GRID) * 4 + 1
assert len(build_input(np.zeros((1, 3)), [YEAR_GRID[0]], 100)) == INPUT_DIM
print("NPE input dimension:", INPUT_DIM)

## 5. Training simulations and estimator

Observation schedules are drawn from a mixture: with probability 0.5 the mask
matches one of the 17 observed grove schedules, and otherwise it is a random
subset of the survey-year grid of random size. The empirical component covers
the schedules present in the study, while the random component exposes the
estimator to a wider range of patterns so the same trained network can also
evaluate alternative surveillance designs (Section 12).

In [ ]:
def sample_mask(rng):
    """Draw one training example's observed-years schedule."""
    if rng.random() < MASK_MIX_P:
        return np.array(sorted(int(y) for y in PATTERNS[rng.integers(len(PATTERNS))]))
    k = rng.integers(2, len(YEAR_GRID) + 1)
    idx = rng.choice(len(YEAR_GRID), size=k, replace=False)
    return np.array(sorted(YEAR_GRID[idx].tolist()))


NPE_PATH = f"xf_npe_posterior_N{N_TRAIN}.pkl"
prior = BoxUniform(low=torch.tensor(LOW, dtype=torch.float32),
                   high=torch.tensor(HIGH, dtype=torch.float32))

if os.path.exists(NPE_PATH):
    print(f"Loading cached estimator from {NPE_PATH}")
    with open(NPE_PATH, "rb") as f:
        posterior = pickle.load(f)
else:
    rng_train = np.random.default_rng(SEED)
    theta_train = LOW + rng_train.random((N_TRAIN, len(PARAMS))) * (HIGH - LOW)
    X = np.empty((N_TRAIN, INPUT_DIM))

    t0 = time.perf_counter()
    for i in range(N_TRAIN):
        N_i = int(rng_train.integers(N_MIN, N_MAX + 1))
        yrs_i = sample_mask(rng_train)
        curve = simulate_curve(*theta_train[i], rng_train, N_i)
        X[i] = build_input(curve[yrs_i], yrs_i, N_i)
        if (i + 1) % 100_000 == 0:
            rate = (i + 1) / (time.perf_counter() - t0)
            print(f"  simulated {i + 1:,}/{N_TRAIN:,}  "
                  f"(ETA {(N_TRAIN - i - 1) / rate / 60:.1f} min)")
    print(f"Simulation completed in {(time.perf_counter() - t0) / 60:.1f} min")

    t1 = time.perf_counter()
    inference = NPE(prior=prior)
    inference.append_simulations(torch.tensor(theta_train, dtype=torch.float32),
                                torch.tensor(X, dtype=torch.float32)).train()
    posterior = inference.build_posterior()
    print(f"Training completed in {(time.perf_counter() - t1) / 60:.1f} min")

    with open(NPE_PATH, "wb") as f:
        pickle.dump(posterior, f)
    print(f"Saved estimator to {NPE_PATH}")

## 6. Posteriors for the 17 groves

One trained estimator, evaluated on each grove's own survey schedule and tree
count.

In [ ]:
t_infer = time.perf_counter()
results = {}
for name, d in plots.items():
    obs_x = build_input(d["obs"], d["years"], d["N"])
    results[name] = posterior.sample(
        (N_POST,), x=torch.tensor(obs_x, dtype=torch.float32),
        show_progress_bars=False).numpy()
print(f"Posterior evaluation for {len(plots)} groves: "
      f"{time.perf_counter() - t_infer:.2f}s\n")

rows = []
for name, s in results.items():
    q = np.percentile(s, [2.5, 50, 97.5], axis=0)
    row = {"plot": name, "N_trees": plots[name]["N"],
           "n_years": len(plots[name]["years"])}
    for j, p in enumerate(PARAMS):
        row[f"{p}_median"], row[f"{p}_lo"], row[f"{p}_hi"] = q[1, j], q[0, j], q[2, j]
    rows.append(row)

summary = pd.DataFrame(rows)
print(summary.round(3).to_string(index=False))
summary.round(4).to_csv("xf_posterior_summary.csv", index=False)

## 7. SI Figure 3 — posteriors vs grove size

In [ ]:
plt.rcParams.update({"font.size": 11, "axes.labelsize": 12, "axes.titlesize": 12,
                     "xtick.labelsize": 10, "ytick.labelsize": 10})

names_by_N = sorted(results, key=lambda n: plots[n]["N"])
Ns = [plots[n]["N"] for n in names_by_N]

ncol = 3
nrow = int(np.ceil(len(PARAMS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 3.5 * nrow))
axes = np.array(axes).ravel()
for ax, p in zip(axes, PARAMS):
    j = PARAMS.index(p)
    med = np.array([np.median(results[n][:, j]) for n in names_by_N])
    lo = np.array([np.percentile(results[n][:, j], 2.5) for n in names_by_N])
    hi = np.array([np.percentile(results[n][:, j], 97.5) for n in names_by_N])
    ax.errorbar(Ns, med, yerr=[med - lo, hi - med], fmt="o", capsize=3,
                color="tab:blue", markersize=5, markeredgecolor="k",
                markeredgewidth=0.5, elinewidth=1)
    ax.set_xlabel("number of trees in plot ($N$)")
    ax.set_ylabel(f"posterior {PARAM_LABELS[p]}")
    ax.spines[["top", "right"]].set_visible(False)
for ax in axes[len(PARAMS):]:
    ax.axis("off")
fig.tight_layout()
fig.savefig("figure_si3.pdf", bbox_inches="tight")
fig.savefig("figure_si3.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Figure 3A — posterior-predictive fits

Posterior draws are resampled until `N_POST` simulated trajectories show an
epidemic (more than one infected tree). Without this filter, groves whose
posterior includes very small $I_{A,0}$ contribute trajectories in which
$I_{A,0}N$ rounds to zero infected trees, and those degenerate
all-susceptible curves dominate the median.

In [ ]:
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8})

rng_ppc = np.random.default_rng(1)
MAX_TRIES = 50 * N_POST

ppc_curves = {}
for name, s in results.items():
    d = plots[name]
    curves_list, tries = [], 0
    while len(curves_list) < N_POST and tries < MAX_TRIES:
        c = simulate_curve(*s[rng_ppc.integers(len(s))], rng_ppc, d["N"])
        if (c[:, 1] + c[:, 2]).max() * d["N"] > 1:
            curves_list.append(c)
        tries += 1
    if not curves_list:
        curves_list = [simulate_curve(*th, rng_ppc, d["N"]) for th in s]
    ppc_curves[name] = np.array(curves_list)

names = list(results.keys())
ncol = 6
nrow = int(np.ceil(len(names) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(2.6 * ncol, 2.1 * nrow),
                         sharex=True, sharey=True)
axes = np.array(axes).ravel()
for i, (ax, name) in enumerate(zip(axes, names)):
    curves = ppc_curves[name]
    med = np.median(curves, axis=0)
    for k, st in enumerate(CANON_STATES):
        ax.plot(YEARS_GRID, curves[:, :, k].T, color=STATE_COLORS[st],
                alpha=0.04, lw=0.8)
        ax.plot(YEARS_GRID, med[:, k], color=STATE_COLORS[st], lw=1.6)
        ax.scatter(plots[name]["years"], plots[name]["obs"][:, k],
                   color=STATE_COLORS[st], edgecolor="k", linewidth=0.4,
                   zorder=3, s=16)
    ax.set_title(name, fontsize=10)
    ax.set_ylim(0, 1)
    ax.spines[["top", "right"]].set_visible(False)
    row, col = divmod(i, ncol)
    if col == 0:
        ax.set_ylabel("prop.")
    if row == nrow - 1 or i + ncol >= len(names):
        ax.set_xlabel("year")
for ax in axes[len(names):]:
    ax.axis("off")

fig.legend(handles=[plt.Line2D([0], [0], color=c, lw=1.6, marker="o",
                               markeredgecolor="k", markersize=5, label=st)
                    for st, c in STATE_COLORS.items()],
           loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.tight_layout()
fig.savefig("figure3a.pdf", bbox_inches="tight")
fig.savefig("figure3a.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Figure 3B — predicted state proportions at year 16

In [ ]:
fig, ax = plt.subplots(figsize=(max(8, 0.55 * len(names)), 4))
width = 0.8 / len(CANON_STATES)
positions = np.arange(len(names))

for k, st in enumerate(CANON_STATES):
    data = [ppc_curves[name][:, YR16, k] for name in names]
    pos = positions + (k - (len(CANON_STATES) - 1) / 2) * width
    bp = ax.boxplot(data, positions=pos, widths=width * 0.9, patch_artist=True,
                    showfliers=False, medianprops=dict(color="k", lw=1.2))
    for box in bp["boxes"]:
        box.set_facecolor(STATE_COLORS[st])
        box.set_alpha(0.7)
        box.set_edgecolor("k")

ax.set_xticks(positions)
ax.set_xticklabels(names, rotation=90)
ax.set_ylim(0, 1)
ax.set_ylabel(f"posterior-predictive proportion of trees (year {YR16})")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, facecolor=STATE_COLORS[st],
                                 alpha=0.7, edgecolor="k", label=st)
                   for st in CANON_STATES],
          loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.15))
fig.tight_layout()
fig.savefig("figure3b.pdf", bbox_inches="tight")
fig.savefig("figure3b.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Figure 3C — forecast variance vs extrapolation horizon

Posterior-predictive variance of the desiccated proportion at year 16, against
the number of years that horizon lies beyond each grove's last survey.

In [ ]:
xref = pd.DataFrame([{
    "plot": name,
    "ID_median": np.median(ppc_curves[name][:, YR16, ID_K]),
    "ID_var": np.var(ppc_curves[name][:, YR16, ID_K]),
    "n_obs": len(plots[name]["years"]),
    "first_obs_yr": int(plots[name]["years"].min()),
    "last_obs_yr": int(plots[name]["years"].max()),
    "yrs_extrapolated": YR16 - int(plots[name]["years"].max()),
} for name in names]).sort_values("ID_var", ascending=False).reset_index(drop=True)

print(xref.round(4).to_string(index=False))
xref.round(5).to_csv("xf_forecast_variance.csv", index=False)

print(f"\nSpearman correlations with posterior-predictive variance of ID at year {YR16}:")
for col in ["n_obs", "yrs_extrapolated", "last_obs_yr"]:
    rho, p = sstats.spearmanr(xref[col], xref["ID_var"])
    print(f"  vs {col:>16s}: rho = {rho:+.3f}, p = {p:.3f}")

In [ ]:
plt.rcParams.update({"font.size": 13, "axes.labelsize": 15, "axes.titlesize": 15,
                     "xtick.labelsize": 12, "ytick.labelsize": 12,
                     "legend.fontsize": 12})

x = xref["yrs_extrapolated"].to_numpy(float)
y = xref["ID_var"].to_numpy(float)

rho, p_rho = sstats.spearmanr(x, y)
slope, intercept, _, _, _ = sstats.linregress(x, y)

x_line = np.linspace(x.min(), x.max(), 100)
y_line = intercept + slope * x_line
dof = len(x) - 2
s_err = np.sqrt(np.sum((y - (intercept + slope * x)) ** 2) / dof)
se_line = s_err * np.sqrt(1 / len(x) + (x_line - x.mean()) ** 2
                          / np.sum((x - x.mean()) ** 2))
ci = sstats.t.ppf(0.975, dof) * se_line

x_jit = x + np.random.default_rng(0).normal(0, 0.07, size=len(x))

fig, ax = plt.subplots(figsize=(6.2, 5))
ax.fill_between(x_line, y_line - ci, y_line + ci, color="0.5", alpha=0.18,
                lw=0, zorder=1)
ax.plot(x_line, y_line, color="0.2", lw=1.8, zorder=2)
sc = ax.scatter(x_jit, y, c=xref["n_obs"].to_numpy(float), cmap="viridis", s=75,
                edgecolor="k", linewidth=0.6, zorder=3)

cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("Number of observations", fontsize=13)
cbar.ax.tick_params(labelsize=11)

ax.set_xlabel("Years extrapolated beyond last observation")
ax.set_ylabel("Posterior variance")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(direction="out", length=4, width=1.0)
ax.text(0.03, 0.97, rf"Spearman $\rho$ = {rho:.2f} (p = {p_rho:.3f})",
        transform=ax.transAxes, ha="left", va="top", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.6", lw=0.8))
fig.tight_layout()
fig.savefig("figure3c.pdf", bbox_inches="tight")
fig.savefig("figure3c.png", dpi=600, bbox_inches="tight")
plt.show()

## 11. Pooled site and Table 4

Following White *et al.* (2020), the 17 groves are aggregated into a single
representative record: for each survey year, tree counts are summed across all
groves observed that year and converted to proportions. The pooled record is
assigned a round $N = 500$ so that it stands in for one hypothetical monitoring
site of representative size.

In [ ]:
AGG_N = 500

agg_years, agg_obs = [], []
for y in YEAR_GRID:
    num, den = np.zeros(3), 0.0
    for d in plots.values():
        if y in d["years"]:
            num += d["N"] * d["obs"][list(d["years"]).index(y)]
            den += d["N"]
    if den > 0:
        agg_years.append(int(y))
        agg_obs.append(num / den)
agg_years = np.array(agg_years)
agg_obs = np.array(agg_obs)
agg_by_year = dict(zip(agg_years.tolist(), agg_obs))

print(f"Pooled site, N = {AGG_N}, years {list(agg_years)}:")
for y, o in zip(agg_years, agg_obs):
    print("  year {:>2d}: ".format(y)
          + ", ".join(f"{s}={v:.3f}" for s, v in zip(CANON_STATES, o)))

In [ ]:
WHITE_2020 = {          # median (95% CrI) reported by White et al. (2020)
    "beta": (17.88, 6.33, 24.88),
    "bA": (0.015, 0.0, 0.44),
    "bD": (0.500, 0.02, 0.98),
    "TA": (1.19, 1.09, 1.27),
    "TD": (1.36, 1.11, 1.59),
    "IA0": (0.0065, 0.003, 0.008),
}

samp_allyrs = posterior.sample(
    (N_POST,), x=torch.tensor(build_input(agg_obs, agg_years, AGG_N),
                              dtype=torch.float32),
    show_progress_bars=False).numpy()

table4 = []
for j, p in enumerate(PARAMS):
    lo, med, hi = np.percentile(samp_allyrs[:, j], [2.5, 50, 97.5])
    w_med, w_lo, w_hi = WHITE_2020[p]
    table4.append({
        "parameter": p,
        "this_study": f"{med:.3g} ({lo:.3g}, {hi:.3g})",
        "white_2020": f"{w_med:.3g} ({w_lo:.3g}, {w_hi:.3g})",
    })

table4 = pd.DataFrame(table4)
print(table4.to_string(index=False))
table4.to_csv("table4.csv", index=False)

## 12. Figure 4A — every survey schedule ranked by forecast variance

All $2^7 - 1 = 127$ non-empty subsets of survey years 4–10 are applied to the
pooled site. For each, the trained estimator is evaluated and the
posterior-predictive variance of the desiccated proportion at year 16 recorded.
No retraining or resimulation is needed, so the whole design space is explored
in one pass.

In [ ]:
MASK_YEARS = np.arange(4, 11)
assert set(MASK_YEARS) == set(agg_years), \
    f"expected survey years 4-10, pooled site has {list(agg_years)}"

ALL_MASKS = [c for r in range(1, len(MASK_YEARS) + 1)
             for c in itertools.combinations(MASK_YEARS, r)]
print(f"{len(ALL_MASKS)} schedules over years {list(MASK_YEARS)}")

rng_mask = np.random.default_rng(2026)
t_mask = time.perf_counter()

mask_records = []
for yrs in ALL_MASKS:
    yrs = np.array(yrs)
    obs_x = build_input(np.array([agg_by_year[int(y)] for y in yrs]), yrs, AGG_N)
    samp = posterior.sample((N_POST,), x=torch.tensor(obs_x, dtype=torch.float32),
                            show_progress_bars=False).numpy()
    id16 = np.array([simulate_curve(*th, rng_mask, AGG_N)[YR16, ID_K]
                     for th in samp])
    mask_records.append({"years": tuple(int(y) for y in yrs),
                         "n_years": len(yrs),
                         "ID_median_yr16": np.median(id16),
                         "ID_var_yr16": np.var(id16)})

mask_df = (pd.DataFrame(mask_records)
           .sort_values("ID_var_yr16", ascending=False).reset_index(drop=True))
mask_df.to_csv("xf_schedule_variance.csv", index=False)
print(f"Evaluated in {time.perf_counter() - t_mask:.1f}s")
print("\nLowest forecast variance by number of observation years:")
print(mask_df.loc[mask_df.groupby("n_years")["ID_var_yr16"].idxmin()]
      .sort_values("n_years")[["n_years", "years", "ID_var_yr16"]]
      .to_string(index=False))

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9, "axes.labelsize": 11, "axes.titlesize": 11,
    "xtick.labelsize": 8, "ytick.labelsize": 9, "axes.linewidth": 0.8,
})

n_masks = len(mask_df)
n_rows = len(MASK_YEARS)
year_row = {int(y): i for i, y in enumerate(MASK_YEARS)}
norm = Normalize(vmin=1, vmax=n_rows)
bar_colors = plt.cm.viridis(norm(mask_df["n_years"].to_numpy()))

fig_w = max(15, 0.155 * n_masks)
fig = plt.figure(figsize=(fig_w, 6.2))
gs = fig.add_gridspec(2, 2, height_ratios=[1.7, 1], width_ratios=[fig_w, 0.35],
                      hspace=0.06, wspace=0.015)
ax_top = fig.add_subplot(gs[0, 0])
ax_mat = fig.add_subplot(gs[1, 0], sharex=ax_top)
cax = fig.add_subplot(gs[:, 1])

ax_top.bar(np.arange(n_masks), mask_df["ID_var_yr16"], color=bar_colors,
           width=0.82, linewidth=0, zorder=2)
ax_top.set_ylabel(r"$\mathrm{Var}(\mathrm{ID}_{16})$")
ax_top.set_ylim(0, mask_df["ID_var_yr16"].max() * 1.12)
ax_top.tick_params(axis="x", labelbottom=False, length=0)
ax_top.tick_params(axis="y", direction="out", length=3.5)
ax_top.spines[["top", "right"]].set_visible(False)

for i in range(n_rows):
    if i % 2 == 0:
        ax_mat.axhspan(i - 0.5, i + 0.5, color="0.96", zorder=0)
for j, row in mask_df.iterrows():
    yrs_in = set(row["years"])
    for y in MASK_YEARS:
        ax_mat.scatter(j, year_row[int(y)], marker="s", s=32,
                       color="#1a2b4c" if int(y) in yrs_in else "#e3e3e3",
                       linewidth=0, zorder=3)

ax_mat.set_yticks(range(n_rows))
ax_mat.set_yticklabels(MASK_YEARS)
ax_mat.set_ylabel("Survey\nyear")
ax_mat.set_xlim(-0.7, n_masks - 0.3)
ax_mat.set_ylim(-0.6, n_rows - 0.4)
ax_mat.set_xticks([])
ax_mat.spines[["top", "right"]].set_visible(False)
ax_mat.tick_params(axis="y", direction="out", length=3.5)

cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=plt.cm.viridis), cax=cax)
cbar.set_label("years observed", fontsize=9)
cbar.ax.tick_params(labelsize=7)
cbar.outline.set_linewidth(0.6)

fig.savefig("figure4a.pdf", bbox_inches="tight")
fig.savefig("figure4a.png", dpi=600, bbox_inches="tight")
plt.show()

## 13. Figure 4B — best schedule at each observation count

In [ ]:
K_SHOWN = [1, 2, 3, 4, 7]
best_by_k = {k: mask_df[mask_df["n_years"] == k]
                .sort_values("ID_var_yr16").iloc[0] for k in K_SHOWN}

fig, axes = plt.subplots(1, len(K_SHOWN), figsize=(4.5 * len(K_SHOWN), 4.2),
                         sharey=True)
for ax, k in zip(axes, K_SHOWN):
    row = best_by_k[k]
    yrs_pat = np.array(row["years"])
    obs_masked = np.array([agg_by_year[int(y)] for y in yrs_pat])
    obs_x = build_input(obs_masked, yrs_pat, AGG_N)
    samp = posterior.sample((N_POST,), x=torch.tensor(obs_x, dtype=torch.float32),
                            show_progress_bars=False).numpy()
    curves = np.array([simulate_curve(*th, rng_mask, AGG_N) for th in samp])
    med = np.median(curves, axis=0)

    for st_k, st in enumerate(CANON_STATES):
        ax.plot(YEARS_GRID, curves[:, :, st_k].T, color=STATE_COLORS[st], alpha=0.03)
        ax.plot(YEARS_GRID, med[:, st_k], color=STATE_COLORS[st], lw=2,
                label=st if k == K_SHOWN[0] else None)
        ax.scatter(yrs_pat, obs_masked[:, st_k], color=STATE_COLORS[st],
                   edgecolor="k", zorder=3, s=30)
    ax.axvline(YR16, color="0.4", ls=":", lw=1)
    ax.set_title(f"$n_{{obs}}$ = {k}: years {list(yrs_pat)}\n"
                 rf"$\mathrm{{Var}}(\mathrm{{ID}}_{{16}})$ = {row['ID_var_yr16']:.2e}",
                 fontsize=10)
    ax.set_xlabel("Year")
    ax.set_ylim(0, 1)
    if ax is not axes[0]:
        ax.tick_params(labelleft=False)
axes[0].set_ylabel("Proportion")
axes[0].legend(fontsize=9, loc="upper left")
fig.tight_layout()
fig.savefig("figure4b.pdf", bbox_inches="tight")
fig.savefig("figure4b.png", dpi=300, bbox_inches="tight")
plt.show()

## 14. Simulation-based calibration

Draw $\theta_k \sim p(\theta)$, pair it with a real grove's survey schedule and
tree count, simulate, and rank the true $\theta_k$ among $L$ posterior draws.

In [ ]:
rng_sbc = np.random.default_rng(2027)
plot_names = list(plots.keys())

sbc_ranks = np.zeros((N_SBC, len(PARAMS)), dtype=int)
t_sbc = time.perf_counter()
for i in range(N_SBC):
    theta_i = LOW + rng_sbc.random(len(PARAMS)) * (HIGH - LOW)
    d = plots[plot_names[rng_sbc.integers(len(plot_names))]]
    curve = simulate_curve(*theta_i, rng_sbc, d["N"])
    obs_x = build_input(curve[d["years"]], d["years"], d["N"])
    samp = posterior.sample((L_SBC,), x=torch.tensor(obs_x, dtype=torch.float32),
                            show_progress_bars=False).numpy()
    sbc_ranks[i] = (samp < theta_i).sum(axis=0)
    if (i + 1) % 1000 == 0:
        print(f"  SBC {i + 1}/{N_SBC}  ({time.perf_counter() - t_sbc:.1f}s)")

sbc_df = pd.DataFrame(sbc_ranks, columns=[f"rank_{p}" for p in PARAMS])
sbc_df.to_csv("xf_sbc_ranks.csv", index=False)
print(f"{N_SBC:,} SBC trials in {time.perf_counter() - t_sbc:.1f}s")

## 15. SI Figure 4A — rank CDFs

In [ ]:
r_grid = np.arange(L_SBC + 1)
p_null = (r_grid + 1) / (L_SBC + 1)
ci_lo, ci_hi = sstats.binom.interval(0.95, N_SBC, p_null)
ci_lo, ci_hi = ci_lo / N_SBC, ci_hi / N_SBC

ncol = 3
nrow = int(np.ceil(len(PARAMS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 3.0 * nrow),
                         sharex=True, sharey=True)
axes = np.array(axes).ravel()
for ax, p in zip(axes, PARAMS):
    ranks_sorted = np.sort(sbc_df[f"rank_{p}"].to_numpy())
    ecdf = np.searchsorted(ranks_sorted, r_grid, side="right") / N_SBC
    ax.fill_between(r_grid / L_SBC, ci_lo, ci_hi, color="0.5", alpha=0.3, zorder=0)
    ax.plot([0, 1], [0, 1], color="k", lw=1, ls="--", zorder=1)
    ax.plot(r_grid / L_SBC, ecdf, color="tab:blue", lw=2, zorder=2)
    ax.set_title(PARAM_LABELS[p])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Normalised rank")
    ax.set_ylabel("Empirical CDF")
for ax in axes[len(PARAMS):]:
    ax.axis("off")
fig.tight_layout()
fig.savefig("figure_si4a.pdf", bbox_inches="tight")
fig.savefig("figure_si4a.png", dpi=300, bbox_inches="tight")
plt.show()

## 16. TARP — joint coverage

In [ ]:
from sbi.diagnostics.tarp import (_run_tarp, check_tarp,
                                  get_posterior_samples_on_batch,
                                  get_tarp_references)

rng_tarp = np.random.default_rng(3037)

tarp_theta = LOW + rng_tarp.random((N_SBC, len(PARAMS))) * (HIGH - LOW)
tarp_x = np.empty((N_SBC, INPUT_DIM))
for i in range(N_SBC):
    d = plots[plot_names[rng_tarp.integers(len(plot_names))]]
    curve = simulate_curve(*tarp_theta[i], rng_tarp, d["N"])
    tarp_x[i] = build_input(curve[d["years"]], d["years"], d["N"])

theta_tarp_t = torch.tensor(tarp_theta, dtype=torch.float32)
x_tarp_t = torch.tensor(tarp_x, dtype=torch.float32)

t_tarp = time.perf_counter()
tarp_chunks = []
for start in range(0, N_SBC, CHUNK):
    end = min(start + CHUNK, N_SBC)
    tarp_chunks.append(get_posterior_samples_on_batch(
        x_tarp_t[start:end], posterior, (L_SBC,), num_workers=1,
        show_progress_bar=False, use_batched_sampling=True))
tarp_posterior_samples = torch.cat(tarp_chunks, dim=1)

ecp, alpha = _run_tarp(tarp_posterior_samples, theta_tarp_t,
                       get_tarp_references(theta_tarp_t), num_bins=30,
                       z_score_theta=True)
atc, ks_pval = check_tarp(ecp, alpha)
print(f"TARP in {time.perf_counter() - t_tarp:.1f}s")
print(f"  area-to-curve: {atc:+.4f}   (>0 overdispersed, <0 underdispersed)")
print(f"  KS p-value:    {ks_pval:.4f}   (>0.05 consistent with calibration)")

## 17. SI Figure 4B — TARP coverage

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Ideal (perfectly calibrated)")
ax.plot(alpha.numpy(), ecp.numpy(), color="tab:blue", lw=2)
ax.set_xlabel(r"Credibility level $\alpha$")
ax.set_ylabel("Expected coverage probability")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
fig.savefig("figure_si4b.pdf", bbox_inches="tight")
fig.savefig("figure_si4b.png", dpi=300, bbox_inches="tight")
plt.show()